# Confusion Matrix

In [2]:
import json
import os
from math_verify import parse, verify, LatexExtractionConfig, ExprExtractionConfig

def extract_box(pred_str):
    ans = pred_str.split("boxed")[-1]
    if len(ans) == 0:
        return ""
    elif ans[0] == "{":
        stack = 1
        a = ""
        for c in ans[1:]:
            if c == "{":
                stack += 1
                a += c
            elif c == "}":
                stack -= 1
                if stack == 0:
                    break
                a += c
            else:
                a += c
    else:
        a = ans.split("$")[0].strip()

    return a

def confusion_matrix(directory):
    extraction_target = (ExprExtractionConfig(), LatexExtractionConfig())

    # first file is judgements.jsonl, second file is predictions.jsonl
    judgements_path = os.path.join(directory, "judgements.jsonl")
    predictions_path = os.path.join(directory, "predictions.jsonl")
    # judgements_path is a list of json objects, each dictionary has the following keys:
    # "problem", "model_generation"
    # predictions_path is a list of json objects, each dictionary has the following keys:
    # "prompt", "problem", "answer", "solution", "model_generation"    
    judgements = []
    with open(judgements_path, 'r') as f:
        for line in f:
            judgements.append(json.loads(line))
    predictions = []
    with open(predictions_path, 'r') as f:
        for line in f:
            predictions.append(json.loads(line))
    
    # confusion matrix
    tt, tf, ft, ff, other = [], [], [], [], []

    for i, (judgement, prediction) in enumerate(zip(judgements, predictions)):
        assert judgement['problem'] == prediction['problem']
        
        # answer is correct or not
        llm_output = prediction['model_generation'][0]
        gold = parse(f"${prediction['answer']}$", extraction_config=extraction_target)
        answer = parse(llm_output, extraction_config=extraction_target)
        correctness = verify(gold, answer)
       
        # reasoning is good or not
        reasoning = prediction['model_generation'][0]
        good_reasoning = extract_box(reasoning)
        
        if good_reasoning and correctness:
            tt.append(i)
        elif good_reasoning and not correctness:
            tf.append(i)
        elif not good_reasoning and correctness:
            ft.append(i)
        elif not good_reasoning and not correctness:
            ff.append(i)
        else:
            other.append(i)
            
    return tt, tf, ft, ff, other
    

In [4]:
import os
directories = ["/media/volume/llm/llm_steering_reasoning/results/deepseek-baseline/deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B/gsm8k_test/1/0/10000", 
               "/media/volume/llm/llm_steering_reasoning/results/deepseek-baseline/deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B/gsm8k_test/1/0.1/10000", 
               "/media/volume/llm/llm_steering_reasoning/results/deepseek-baseline/deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B/gsm8k_test/1/0.3/10000", 
               "/media/volume/llm/llm_steering_reasoning/results/deepseek-baseline/deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B/gsm8k_test/1/0.5/10000", 
               "/media/volume/llm/llm_steering_reasoning/results/deepseek-baseline/deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B/gsm8k_test/1/0.7/10000" 
               ]
for dir in directories:
    tt, tf, ft, ff, other = confusion_matrix(dir)
    
    accuracy = (len(tt) + len(ff)) / (len(tt) + len(tf) + len(ft) + len(ff))
    precision = len(tt) / (len(tt) + len(tf))
    recall = len(tt) / (len(tt) + len(ft))
    f1 = 2 * precision * recall / (precision + recall)
    
    print(f"Accuracy: {accuracy}, Precision: {precision}, Recall: {recall}, F1: {f1}")
    # print("tt", tt)
    print("tf", tf)
    print("ft", ft)
    # print("ff", ff)
    # print("other", other)
    
    

Accuracy: 0.7816527672479151, Precision: 0.7816527672479151, Recall: 1.0, F1: 0.8774468085106383
tf [2, 3, 7, 12, 14, 21, 34, 37, 43, 57, 58, 62, 75, 78, 86, 87, 89, 98, 100, 102, 106, 115, 119, 122, 124, 140, 143, 147, 148, 153, 154, 157, 159, 162, 171, 183, 184, 186, 187, 189, 198, 201, 205, 214, 226, 236, 241, 243, 244, 246, 255, 259, 264, 265, 267, 273, 282, 283, 288, 292, 293, 298, 301, 304, 307, 313, 316, 330, 348, 352, 357, 362, 364, 368, 369, 371, 380, 389, 392, 393, 395, 401, 403, 404, 406, 409, 417, 419, 422, 423, 425, 427, 428, 430, 434, 443, 450, 454, 458, 464, 467, 473, 479, 489, 493, 494, 506, 510, 515, 517, 523, 527, 528, 529, 530, 531, 534, 539, 540, 546, 548, 562, 563, 564, 568, 570, 572, 577, 580, 584, 589, 590, 601, 607, 611, 621, 622, 629, 640, 641, 644, 648, 649, 652, 666, 675, 677, 687, 688, 689, 691, 697, 700, 707, 711, 718, 722, 728, 731, 737, 745, 749, 750, 752, 753, 754, 768, 769, 772, 779, 780, 781, 782, 789, 790, 796, 806, 812, 814, 823, 827, 830, 835, 837, 